# Cats vs Dogs — Exploratory Data Analysis**MLOps (S1-25_AIMLCZG523) — Assignment 2, M1**Checks the dataset before it reaches the training pipeline: class balance,image dimensions, aspect ratios, corrupt files, and the effect of theaugmentation pipeline.

In [ ]:
import sys, jsonfrom collections import Counterfrom pathlib import Pathsys.path.insert(0, "..")import matplotlib.pyplot as pltimport numpy as npfrom PIL import Imagefrom src.config import load_configfrom src.data.preprocess import is_valid_image, split_indicescfg = load_config("../params.yaml")RAW = Path("..") / cfg.data.raw_dir / "PetImages"PROCESSED = Path("..") / cfg.data.processed_dirplt.rcParams["figure.dpi"] = 110print("raw       :", RAW.resolve())print("processed :", PROCESSED.resolve())

## 1. Class balance in the raw corpus

In [ ]:
raw_counts = {c: len(list((RAW / c).glob("*.jpg"))) for c in ("Cat", "Dog") if (RAW / c).is_dir()}print(raw_counts)fig, ax = plt.subplots(figsize=(5, 3.5))ax.bar(raw_counts.keys(), raw_counts.values(), color=["#4C9F70", "#D97757"])ax.set(ylabel="images", title="Raw class balance")for i, (k, v) in enumerate(raw_counts.items()):    ax.text(i, v, f"{v:,}", ha="center", va="bottom")plt.tight_layout(); plt.show()

The corpus is perfectly balanced (12,500 per class), so accuracy is a fairheadline metric and no class weighting is needed.

## 2. Corrupt / unreadable files

In [ ]:
# The corpus ships a few zero-byte and truncated JPEGs that crash training# loaders. preprocess.is_valid_image() filters them before the split.SCAN = 400report = {}for cls in raw_counts:    paths = sorted((RAW / cls).glob("*.jpg"))[:SCAN]    bad = [p.name for p in paths if not is_valid_image(p)]    report[cls] = bad    print(f"{cls}: {len(bad)} corrupt in the first {len(paths)} files -> {bad[:5]}")

## 3. Image dimensions and aspect ratios

In [ ]:
SAMPLE = 300sizes = []for cls in raw_counts:    for p in sorted((RAW / cls).glob("*.jpg"))[:SAMPLE]:        if not is_valid_image(p):            continue        with Image.open(p) as im:            sizes.append((im.width, im.height, cls))widths  = np.array([s[0] for s in sizes])heights = np.array([s[1] for s in sizes])ratios  = widths / heightsprint(f"width  : min={widths.min()}  median={np.median(widths):.0f}  max={widths.max()}")print(f"height : min={heights.min()}  median={np.median(heights):.0f}  max={heights.max()}")print(f"aspect : min={ratios.min():.2f}  median={np.median(ratios):.2f}  max={ratios.max():.2f}")fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))axes[0].hist(widths, bins=40, color="#4C9F70");  axes[0].set(title="Width (px)", xlabel="px")axes[1].hist(heights, bins=40, color="#D97757"); axes[1].set(title="Height (px)", xlabel="px")axes[2].hist(ratios, bins=40, color="#6B7FD7")axes[2].axvline(1.0, color="k", ls="--", lw=1, label="square")axes[2].set(title="Aspect ratio (w/h)", xlabel="ratio"); axes[2].legend()plt.tight_layout(); plt.show()

Images vary widely in size and aspect ratio, which is why the pipeline resizeseverything to a fixed **224×224 RGB** — the standard input for the CNNarchitectures used here. The resize is not aspect-preserving; at these ratiosthe distortion is mild and the augmentation crop absorbs most of it.

## 4. Sample images

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))for row, cls in enumerate(raw_counts):    paths = [p for p in sorted((RAW / cls).glob("*.jpg"))[:40] if is_valid_image(p)][:6]    for col, p in enumerate(paths):        with Image.open(p) as im:            axes[row, col].imshow(im.convert("RGB"))        axes[row, col].set_title(f"{cls} {im.size}", fontsize=8)        axes[row, col].axis("off")plt.tight_layout(); plt.show()

## 5. The processed splits

In [ ]:
stats_path = PROCESSED / "dataset_stats.json"if stats_path.exists():    stats = json.loads(stats_path.read_text())    print(json.dumps(stats, indent=2))    splits = stats["splits"]    labels = list(splits)    cats = [splits[s].get("cat", 0) for s in labels]    dogs = [splits[s].get("dog", 0) for s in labels]    x = np.arange(len(labels))    fig, ax = plt.subplots(figsize=(6, 3.5))    ax.bar(x - 0.2, cats, 0.4, label="cat", color="#4C9F70")    ax.bar(x + 0.2, dogs, 0.4, label="dog", color="#D97757")    ax.set_xticks(x, labels); ax.set(ylabel="images", title="Processed split sizes")    ax.legend(); plt.tight_layout(); plt.show()else:    print("Run `python -m src.data.preprocess` first.")

Both classes are split independently with the same seed, so every split staysbalanced — the val and test accuracies are directly comparable.

## 6. Split determinism

In [ ]:
# The pipeline must produce identical splits on every run, or DVC caching and# experiment comparison become meaningless.a = split_indices(1000, 0.8, 0.1, 0.1, seed=42)b = split_indices(1000, 0.8, 0.1, 0.1, seed=42)c = split_indices(1000, 0.8, 0.1, 0.1, seed=7)print("same seed reproduces the split :", a == b)print("different seed changes it      :", a != c)print("sizes (train/val/test)         :", [len(s) for s in a])print("disjoint and complete          :",      set(a[0]) | set(a[1]) | set(a[2]) == set(range(1000)))

## 7. Augmentation preview

In [ ]:
from src.data.dataset import build_eval_transform, build_train_transformfrom src.config import NORM_MEAN, NORM_STDtrain_tf = build_train_transform(cfg.data.image_size, cfg.augment)eval_tf  = build_eval_transform(cfg.data.image_size)def denorm(t):    arr = t.permute(1, 2, 0).numpy() * np.array(NORM_STD) + np.array(NORM_MEAN)    return np.clip(arr, 0, 1)sample = next(p for p in sorted((RAW / "Dog").glob("*.jpg"))[:40] if is_valid_image(p))with Image.open(sample) as im:    rgb = im.convert("RGB")fig, axes = plt.subplots(1, 6, figsize=(15, 2.9))axes[0].imshow(denorm(eval_tf(rgb))); axes[0].set_title("eval (deterministic)", fontsize=9)axes[0].axis("off")for i in range(1, 6):    axes[i].imshow(denorm(train_tf(rgb))); axes[i].set_title(f"augmented {i}", fontsize=9)    axes[i].axis("off")plt.suptitle("Train-time augmentation vs the deterministic eval transform", y=1.06)plt.tight_layout(); plt.show()

Only the training split is augmented (random-resized crop, horizontal flip,±15° rotation, colour jitter). Validation, test **and the production API** allshare the single deterministic `build_eval_transform` — that shared function iswhat keeps serving-time preprocessing from drifting away from training.

## 8. Pixel intensity distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))for cls, color in zip(raw_counts, ["#4C9F70", "#D97757"]):    values = []    for p in [q for q in sorted((RAW / cls).glob("*.jpg"))[:60] if is_valid_image(q)][:30]:        with Image.open(p) as im:            values.append(np.asarray(im.convert("RGB").resize((64, 64))).ravel())    ax.hist(np.concatenate(values), bins=60, alpha=0.55, density=True, label=cls, color=color)ax.set(xlabel="pixel intensity", ylabel="density", title="Pixel intensity by class")ax.legend(); plt.tight_layout(); plt.show()

The two classes have near-identical intensity distributions — there is notrivial colour/brightness shortcut, so the model has to learn actual shape andtexture features. This is why a from-scratch CNN needs several epochs beforevalidation accuracy moves off chance level.---## Findings that shaped the pipeline| Observation | Pipeline response ||---|---|| Perfectly balanced classes | Plain accuracy is a fair headline metric; no class weighting || Corrupt / zero-byte JPEGs present | `is_valid_image()` filters them before the split || Highly variable dimensions | Fixed 224×224 RGB resize || No colour/intensity shortcut | Augmentation + enough epochs are genuinely needed || Splits must be reproducible | Seeded `split_indices`, verified in `tests/test_preprocess.py` |